# PGABL - Setup Verify (Tahap 0)

**Proyek**: Fine-tuned Chatbot Tim Legal berbasis RAG
**Peserta**: Nazhif Setya Nugroho
**Tujuan Notebook**: verifikasi environment Colab T4 siap untuk Tahap 1-6 (dependency terinstall, GPU tersedia, HF/WANDB token verified, 4 HF checkpoint accessible).

## Yang perlu disiapkan SEBELUM Run All:

1. **Runtime → Change runtime type → T4 GPU** (bukan CPU / TPU).
2. **Colab Secrets** (menu kunci 🔑 di sidebar kiri) - tambahkan 2 secrets:
   - `HF_TOKEN` (scope **Write** dari https://huggingface.co/settings/tokens)
   - `WANDB_API_KEY` (dari https://wandb.ai/authorize)
   - Toggle **Notebook access** ON untuk keduanya.
3. **Mount Google Drive** (akan dilakukan di Cell 4).

## Menjalankan

**Runtime → Run all.** Total waktu ~3-5 menit.

**⚠️ Cell 5 akan meminta kamu restart runtime** (Runtime → Restart runtime, atau pilih `Runtime → Restart session and run all` supaya otomatis). Setelah restart, jalankan `Runtime → Run all` sekali lagi — Cell 5 akan cepat (package sudah terinstall), lanjut ke Cell 6 tanpa ImportError.

## Catatan: ragas ditunda

`ragas` (evaluasi RAG) **tidak di-install di Tahap 0** karena ada compat issue dgn `langchain-community` terbaru (July 2026): ragas versi yg tersedia mencoba `from langchain_community.chat_models.vertexai import ChatVertexAI`, tapi module `chat_models.vertexai` sudah dihapus di langchain-community ≥ 0.3 (dipindah ke package `langchain-google-vertexai`). Fix akan ditangani fokus di **Tahap 4 (Evaluasi)** dengan pin ragas + langchain compat spesifik. Untuk Tahap 0-3 kita cukup butuh `rouge-score` (untuk GRPO correctness reward di Tahap 2).

## Cell 1: Detect environment (Colab vs local)

In [ ]:
import sys, os, platform

def detect_env():
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"

ENV = detect_env()
print(f"Environment: {ENV}")
print(f"Python: {sys.version.split()[0]}")
print(f"OS: {platform.system()} {platform.release()}")
assert ENV == "colab", "Notebook ini didesain untuk Colab T4. Untuk local dev, pakai scripts/*.py."

## Cell 2: Load secrets (HF_TOKEN + WANDB_API_KEY)

Sumber: Colab Secrets (menu kunci 🔑). **Bukan hardcode di notebook** - kalau ketahuan hardcoded = auto-reject rubric.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
WANDB_API_KEY = userdata.get('WANDB_API_KEY')

assert HF_TOKEN, "HF_TOKEN tidak ditemukan di Colab Secrets. Set di menu kunci sidebar kiri."
assert WANDB_API_KEY, "WANDB_API_KEY tidak ditemukan di Colab Secrets. Set di menu kunci sidebar kiri."

# Set sebagai env var supaya library yg baca os.environ juga dapet
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN
os.environ['WANDB_API_KEY'] = WANDB_API_KEY

print(f"HF_TOKEN loaded (starts with: {HF_TOKEN[:6]}...)")
print(f"WANDB_API_KEY loaded (starts with: {WANDB_API_KEY[:6]}...)")

## Cell 3: nvidia-smi - verify T4 GPU tersedia

Target: `Tesla T4` dengan free memory `≥ 14 GB`.

In [ ]:
!nvidia-smi

## Cell 4: Mount Google Drive

Semua checkpoint SFT/GRPO + ChromaDB persistent akan disimpan di Drive → session-safe (bertahan disconnect Colab).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Buat folder skeleton di Drive (mirror struktur lokal)
DRIVE_ROOT = '/content/drive/MyDrive/PGABL'
SUBFOLDERS = [
    'data/raw', 'data/processed', 'data/test_set',
    'chroma_db',
    'models', 'checkpoints/sft', 'checkpoints/grpo',
    'outputs/wandb', 'outputs/eval_reports', 'outputs/samples',
    'outputs/ui_evidence', 'outputs/finetune_evidence', 'outputs/setup_evidence',
]
for sf in SUBFOLDERS:
    os.makedirs(f'{DRIVE_ROOT}/{sf}', exist_ok=True)

print(f"Drive mounted. PGABL root: {DRIVE_ROOT}")
print(f"Subfolders created: {len(SUBFOLDERS)}")
!ls -la $DRIVE_ROOT

## Cell 5: Install stack

Unsloth 2026.7.x yang mengurus dependency `transformers` / `trl` / `datasets` / `accelerate` / `peft` / `bitsandbytes` — **jangan pin manual** karena Colab July 2026 butuh transformers ≥ 4.51.3 (untuk `CompileConfig`), datasets ≥ 3.4.1, trl ≥ 0.18.2. Pin manual dari sebelum 2026 akan bikin `ImportError` di Cell 6.

**Waktu**: ~3-5 menit (first run). ~30 detik (re-run setelah restart).

**⚠️ Warning `pip's dependency resolver ... dependency conflicts` di akhir output = WAJAR** (Colab pre-install banyak library dengan version constraint bersaing). Yang penting: tidak ada fatal error di baris terakhir + `=== Install selesai ===` muncul.

**⚠️ WAJIB RESTART RUNTIME sebelum Cell 6** (dijelaskan di sel berikutnya).

In [ ]:
# 1. Unsloth first — bawa transformers/trl/datasets/accelerate/peft/bitsandbytes versi kompatibel
#    JANGAN pin transformers/trl/datasets manual — Colab July 2026 punya Unsloth 2026.7.x
#    yang butuh transformers >= 4.51.3 (untuk CompileConfig)
!pip install -q "unsloth[colab-new]"

# 2. Upgrade Pillow (pdfplumber >= 0.11 butuh Pillow >= 12.2)
!pip install -q -U "Pillow>=12.2"

# 3. RAG: ChromaDB + Sentence-Transformers + FlagEmbedding
!pip install -q chromadb sentence-transformers FlagEmbedding

# 4. PDF processing
!pip install -q pypdf pdfplumber

# 5. Evaluation - rouge-score untuk GRPO correctness reward (Tahap 2)
#    NOTE: ragas ditunda ke Tahap 4 karena compat issue langchain-community
#    (ragas versi tersedia import 'langchain_community.chat_models.vertexai' yg sudah dihapus)
!pip install -q rouge-score

# 6. Retrieval helpers
!pip install -q rank-bm25

# 7. Advanced RAG
!pip install -q duckduckgo-search langdetect

# 8. Config & logging (wandb + yaml)
!pip install -q pyyaml wandb

# 9. Requirements freezer (untuk Tahap 6)
!pip install -q pipreqs

# NOTE: gradio + accelerate + peft + transformers + trl + datasets di-handle Unsloth di step 1
print("\n" + "="*60)
print("=== Install selesai ===")
print("="*60)
print("\n[WAJIB] RESTART RUNTIME sekarang, sebelum jalankan Cell 6.")
print("  Menu: Runtime -> Restart runtime  (shortcut: Ctrl+M .)")
print("  Atau: Runtime -> Restart session and run all (otomatis re-run semua cell)")
print("\nKenapa: Colab sudah load transformers versi lama ke memory.")
print("Install baru tidak menggantikan module yang sudah in-memory.")
print("Restart bikin Python re-load semua module dari filesystem (versi baru).")

## ⚠️ WAJIB: Restart Runtime setelah Cell 5

Kalau kamu belum restart setelah Cell 5, **restart sekarang**:

- Menu: `Runtime → Restart session and run all`  (paling efisien — otomatis re-run semua cell)
- Atau: `Runtime → Restart runtime` (Ctrl+M .) lalu `Runtime → Run all`

Cell 5 akan re-run tapi cepat (~30 detik — package sudah terinstall, cuma cek version).

**Kenapa perlu restart**: Colab pre-load beberapa module (mis. `transformers` versi < 4.51). Install `unsloth` di Cell 5 upgrade `transformers` di filesystem, tapi TIDAK menggantikan module yang sudah in-memory. Restart bikin Python re-load semua module dari filesystem.

**Gejala kalau belum restart**: `ImportError: cannot import name 'CompileConfig' from 'transformers'` (atau error import serupa) di Cell 6.

## Cell 6: Verify imports + CUDA availability

Kalau ada `ImportError` di sini SETELAH restart → install Cell 5 gagal (bukan cache in-memory). Cek warning di Cell 5 (bisa jadi Unsloth install fail — biasanya karena network glitch, retry aja).

In [ ]:
import torch

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "CUDA tidak tersedia. Runtime → Change runtime type → T4 GPU"
print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"CUDA capability: {torch.cuda.get_device_capability(0)}")

# Core imports (Tahap 2 - Fine-tuning)
from unsloth import FastLanguageModel
from trl import SFTTrainer, GRPOTrainer, SFTConfig, GRPOConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, PeftModel

# RAG imports (Tahap 3 - RAG Pipeline)
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

# PDF imports (Tahap 1 - PDF prep)
import pypdf, pdfplumber

# Eval imports (Tahap 2 - GRPO correctness reward)
# NOTE: ragas ditunda ke Tahap 4 (compat issue - lihat markdown intro notebook)
from rouge_score import rouge_scorer

# Advanced RAG imports (Tahap 3 Advanced)
from duckduckgo_search import DDGS
from langdetect import detect, detect_langs

# UI (Tahap 5 - Gradio)
import gradio as gr

# Config
import yaml

print("\n=== All imports OK ===")
print(f"transformers: {__import__('transformers').__version__}")
print(f"trl: {__import__('trl').__version__}")
print(f"datasets: {__import__('datasets').__version__}")
print(f"chromadb: {chromadb.__version__}")
print(f"gradio: {gr.__version__}")
print("ragas: DEFERRED to Tahap 4 (langchain-community compat issue)")

## Cell 7: HuggingFace + Weights & Biases login

Login pakai token dari Cell 2 - test bahwa token bisa dipakai.

In [ ]:
from huggingface_hub import login as hf_login, HfApi

# HF login
hf_login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi()
hf_user = api.whoami()
print(f"HuggingFace login OK: {hf_user['name']} ({hf_user.get('type', 'user')})")
print(f"HF plan: {hf_user.get('plan', 'free')}")

# W&B login
import wandb
wandb.login(key=WANDB_API_KEY)
print(f"\nWeights & Biases login OK")

## Cell 8: Verify 4 HuggingFace checkpoint accessible

Cek 4 resource yang akan dipakai Tahap 2-3 masih hidup + accessible.

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

REQUIRED_MODELS = [
    ('unsloth/Llama-3.2-3B-Instruct', 'model'),  # SLM base (K1)
    ('BAAI/bge-m3', 'model'),                      # Embedder (K2)
    ('BAAI/bge-reranker-v2-m3', 'model'),          # Reranker (K2 Advanced)
    ('Ichsan2895/alpaca-gpt4-indonesian', 'dataset'),  # SFT dataset (K1)
]

print("Verifying HF checkpoints...\n")
for repo_id, repo_type in REQUIRED_MODELS:
    try:
        info = api.repo_info(repo_id=repo_id, repo_type=repo_type)
        print(f"  OK  {repo_type:8s}  {repo_id}")
        print(f"        last_modified: {info.last_modified}")
    except Exception as e:
        print(f"  FAIL {repo_type:8s}  {repo_id}")
        print(f"        error: {e}")

print("\n=== Verify HF checkpoints selesai ===")

## Cell 9: Save setup evidence

Simpan `nvidia-smi` + `pip freeze` ke Drive untuk audit trail.

In [ ]:
# Simpan output nvidia-smi ke file
!nvidia-smi > /content/drive/MyDrive/PGABL/outputs/setup_evidence/nvidia_smi.txt 2>&1

# Simpan versi dependency
!pip freeze > /content/drive/MyDrive/PGABL/outputs/setup_evidence/pip_freeze.txt

# Verify saved
!ls -la /content/drive/MyDrive/PGABL/outputs/setup_evidence/
print("\n=== Setup evidence saved to Drive ===")

## ✅ Tahap 0 selesai

Kalau semua cell di atas hijau (tidak ada assertion error / import error), Tahap 0 ✅ DONE.

**Next**: Tahap 1 - Data Preparation.

Ringkasan Tahap 1:
- 1a: Load `Ichsan2895/alpaca-gpt4-indonesian` (2 kolom: input, output) + EDA + split 90/10 stratified + apply Llama-3 chat template + print output ter-format
- 1b: Loader/cleaner/chunker 4 PDF → `data/processed/pdfs/*/chunks.json`
- 1c: Kurasi 30-50 Q&A test-set → `data/test_set/legal_qa_testset.jsonl`

**Catatan deferred item**: `ragas` (evaluasi RAG) akan di-install + fix compat issue di **Tahap 4 (Evaluasi)** dengan investigasi fokus (kemungkinan pin ragas + langchain versi tertentu, atau install `langchain-google-vertexai` shim).